# 체크포인터를 사용한 상태 관리

- 랭그래프의 체크포인터(checkpointer) 시스템은 '상태 영속성'과 '오류 복구'를 위한 핵심 기능. 

- LangGraph의 그래프는 각 노드를 실행할 때마다 State(상태 객체)를 주고받으며 업데이트합니다. 
- checkpointer는 이 State를 매 super-step(각 노드 실행 후)마다 스냅샷으로 저장하는 역할을 합니다. (상태영속성)


- checkpointer 를 사용하여  '시스템 오류 복구' 시에도 마지막 체크포인트부터 재시작을 할 수 있습니다. 
- 이전 에는 대화의 이력을 저장하려면 별도로 코드 작업을 해야 했는데, 랭그래프에서는 이런 작업을 설정 한 줄로 끝낼 수 있도록 지원하고 있습니다.

- 사용 방법
    1. 체크포인터 설정
    2. 그래프에 체크포인터 연결
    3. thread_id 로 상태 관리 (나머지는 랭그래프에서 알아서 동작시켜줌)

- thread_id가 하는 일
    - thread_id는 하나의 "대화 세션(또는 실행 흐름)"을 식별하는 키입니다.
    - checkpointer는 내부적으로 (thread_id, checkpoint_id) 조합으로 상태 스냅샷들을 저장소(메모리, SQLite, Postgres 등)에 쌓아둡니다.
    - 같은 thread_id로 그래프를 다시 호출하면, checkpointer가 가장 최근 checkpoint를 찾아서 그 지점부터 State를 복원한 뒤 이어서 실행합니다.


In [ ]:
# 예시
"""
from langgraph.checkpoint.sqlite import SqliteSaver
# pip install langgraph-checkpoint-sqlite 필요

from langgraph.graph import StateGraph

# 1. 체크포인터 설정
checkpointer = SqliteSaver.from_conn_string(":memory:")

# 2. 그래프에 체크포인터 연결
app = StateGraph(state_schema).compile(checkpointer = checkpointer) ⭐️

# 3. 스레드 ID로 상태 관리
config = {"configurable": {"thread_id": "thread-1"}}
result = app.invoke(input_date, config=config) ⭐
"""
None

In [ ]:
# 기본적으로 제공하는 체크포인터는 다음과 같습니다.

# • BaseCheckpointSaver : 추상 기본 클래스
# • InMemorySaver : 메모리 기반 구현
# • SQLiteSaver, PostgresSaver, ... : 영구 저장소 구현

"""
 우리는 InMemorySaver를 사용하는 간단한 예제를 만들겠습니다.
"""
None

# import

In [3]:
from dotenv import load_dotenv
print(load_dotenv())

from typing import Dict, Any, Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json

True


In [4]:
# InMemorySaver 임포트
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
# InMemorySaver : 랭그래프의 체크포인트 시스템 중 가장 기본적인 구현체입니다. 메모리에 상태를 저장하는 방식으로, 
#   다음과 같은 특징을 가집니다.

# • 휘발성 : 프로그램이 종료되면 데이터가 사라집니다.
# • 빠른 속도: 메모리 접근이므로 매우 빠릅니다.
# • 개발/테스트용 : 주로 프로토타이핑과 테스트에 사용됩니다.

# 프로덕션에서는 SQLiteSaver, PostgresSaver 등으로 교체하여 영구 저장할 수 있습니다.

# 그래프 상태 정의

In [ ]:
"""
② 그래프 상태 정의: 그래프 전체적으로 공유하는 데이터입니다. 
 사용자 입력 메시지, 사용자 이름, 사용자가 좋아하는 것, 싫어하는 것, 최종 답변을 상태로 저장합니다. 
 체크포인트는 이런 상태에 대한 이력을 자동으로 관리합니다.
"""
None

In [3]:
# ② 그래프 상태 정의
# 사용자 입력 메시지, 사용자 이름, 사용자 좋아하는것, 싫어하는 것, 최종답변 을 저장
# 체크포인트는 이런 상태에 대한 이력을 자동 관리한다.
class MemoryBotState(BaseModel):
    user_message: str = Field(default="", description="사용자 입력 메시지")
    user_name: str = Field(default="", description="사용자 이름")
    user_preferences: Dict[str, Any] = Field(
        default_factory=dict,
        # default_factory 는 기본값을 '값' 이 아니라 '함수' 로 지정
        # 좋아하는 것 "likes": [...],   싫어하는 것 "dislikes": [...]
        description="사용자 선호도",
    )
    response: str = Field(default="", description="최종 응답")

# ↑ 체크포인터는 위 상태에 대한 이력을 자동으로 관리하게 된다!    

# 메시지 처리 노드

In [ ]:
"""
② 메시지 처리 노드: 순수한 비즈니스 로직만 담당하는 노드입니다. 
사용자의 메시지를 분석하여 상태를 업데이트하는 역할을 합니다. 

★ InMemorySaver가 노드 실행 전에 자동으로 이전 상태를 로드하며, 노드 실행 후에 자동으로 새 상태를 저장합니다. 
그러므로 개발자가 별도로 대화 이력관리를 하지 않아도 됩니다.
"""
None

In [26]:
# LangChain LLM 초기화
llm = ChatOpenAI(model="gpt-4o")

# ③ 메시지 처리 노드: 사용자의 메시지를 분석하여 상태를 업데이트하는 역할.
def process_message(state: MemoryBotState) -> Dict[str, Any]:
    message = state.user_message
    user_name = state.user_name
    preferences = state.user_preferences.copy()   # 사본!

    # 시스템 프롬프트
    system_prompt = f"""
당신은 사용자의 정보를 기억하는 메모리 봇입니다.
현재 기억하고 있는 정보:
- 사용자 이름: {user_name if user_name else "모름"}
- 좋아하는 것: {preferences.get("likes", [])}
- 싫어하는 것: {preferences.get("dislikes", [])}

사용자 메시지를 분석하여 다음 JSON 형태로 응답하세요:
{{
  "response": "사용자에게 줄 응답 메시지",
  "new_name": "새로 알게 된 이름 (없으면 null)",
  "new_likes": ["새로 알게 된 좋아하는 것들"],
  "new_dislikes": ["새로 알게 된 싫어하는 것들"]
}}
"""

    messages = [SystemMessage(content=system_prompt), HumanMessage(content=message)]

    response = llm.invoke(messages)
    print('✅ 응답확인:', response.content)  # 확인용    

    result = json.loads(response.content) # 파이썬객체 <= JSON 

    # 위 result 에 담긴 새로운 정보에 따라 상태 업데이트 
    result.get('new_name') and (user_name := result['new_name'])
    result.get('new_likes') and preferences.setdefault("likes", []).extend(result['new_likes'])    
    result.get('new_dislikes') and preferences.setdefault("dislikes", []).extend(result['new_dislikes'])   

    bot_response = result.get("response", "죄송해요, 희준해서 죄송해요")

    return {
        "response": bot_response,
        "user_name": user_name,
        "user_preferences": preferences,  # dict 사본을 세팅해야 상태 업데이트 됨!  (마치 React!)
    }


In [19]:
aaa = {}
aaa.setdefault("likes", [])
aaa

{'likes': []}

In [20]:
aaa = {'likes': ['용현', '태현']}
aaa.setdefault("likes", [])
aaa

{'likes': ['용현', '태현']}

In [21]:
aaa = {'likes': ['용현', '태현']}
aaa.setdefault("likes", []).extend(['정준', '희준'])
aaa

{'likes': ['용현', '태현', '정준', '희준']}

# 메모리 봇 그래프 생성

In [27]:
# ④ 메모리 봇 그래프 생성
def create_memory_bot_graph():
    # ⑤ InMemorySaver로 자동 메모리 관리 (아래 설명)
    checkpointer = InMemorySaver()

    workflow = StateGraph(MemoryBotState)

    workflow.add_node("process_message", process_message)

    workflow.add_edge(START, "process_message")
    workflow.add_edge("process_message", END)

    # ⑥ checkpointer와 함께 컴파일 (아래 설명)
    return workflow.compile(checkpointer=checkpointer)

## InMemorySaver로 자동 메모리 관리

In [ ]:
# InMemorySaver로 자동 메모리 관리: InMemorySaver의 내부 구조를 매우 단순하게 '표현'하면 다음과 같습니다. 
# 저장소로 사용할 딕셔너리 (storage)를 만들고 거기에 thread_id별로 데이터를 쌓고, 
# 가져올 때도 thread_id를 기준으로 가져옵니다.

"""
class InMemorySaver:
    def __init__(self):
        self.storage = {}  # {thread_id: {checkpoint_id: state}}

    def put(self, config, checkpoint):
        thread_id = config['configurable']['thread_id']
        self.storage[thread_id] = checkpoint

    def get(self, config):
        thread_id = config['configurable']['thread_id']
        return self.storage.get(thread_id)
"""
None


## checkpointer 와 함께 컴파일

In [ ]:
# checkpointer와 함께 컴파일: compile() 메서드에 체크포인터를 전달하면 다음과 같은 일이 발생합니다.

# • 각 노드를 래핑: 체크포인터가 각 노드 실행을 감싸는 래퍼 생성
# • 자동 체크포인트: 노드 실행 전후에 자동으로 체크포인트 생성
# • 상태 병합: 이전 상태와 새 입력을 자동으로 병합


# InMemorySaver 사용을 위한 config 설정

In [ ]:
# InMemorySaver 사용을 위한 config 설정 : 체크포인트를 사용하려면 thread_id가 필요합니다. 
# thread_id 설정은 그래프 실행 시 설정으로 추가하면 됩니다. 
# config는 '매 실행마다 전달'되어야 합니다. 

# InMemorySarver는 config에 있는 thread_id를 확인하여 어디에 있는 데이터를 불러올지, 
# 어디에 상태를 저장해야 할지 알 수 있게 됩니다.


# 실행

In [28]:
def main():
    print("=== InMemorySaver 메모리 봇 테스트 ===\n")

    app = create_memory_bot_graph()
    thread_id = "orange_123"  # ⑦ thread_id - 세션 식별자

    # 테스트 대화
    conversations = [
        "안녕하세요!",
        "내 이름은 감귤이야",  # 이름 정보
        "김치찜을 좋아해",   # '좋아하는 것' 정보
        "해삼은 싫어해",    # '싫어하는 것' 정보
        # ↓ 메모리의 내용이 답변이 되어야 한다!
        "내 이름이 뭐였지?",
        "내가 좋아하는 것과 싫어하는 것은?",
    ]

    for i, message in enumerate(conversations, 1):
        print(f"[{i}] 사용자: {message}")

        # InMemorySaver 사용을 위한 config 설정
        config = {"configurable": {"thread_id": thread_id}}

        result = app.invoke({"user_message": message}, config)  # invoke() 에 config 전달!
        
        print(f"[{i}] 챗봇: {result['response']}")
        print(
            f"메모리: 이름={result.get('user_name', '없음')}, "
            f"호불호: {result.get('user_preferences', {})}\n"
        )            


if __name__ == "__main__":
    main()


=== InMemorySaver 메모리 봇 테스트 ===

[1] 사용자: 안녕하세요!
✅ 응답확인: {
  "response": "안녕하세요! 만나서 반가워요. 이름이나 좋아하는 것에 대해 알려주시면 기억할게요.",
  "new_name": null,
  "new_likes": [],
  "new_dislikes": []
}
[1] 챗봇: 안녕하세요! 만나서 반가워요. 이름이나 좋아하는 것에 대해 알려주시면 기억할게요.
메모리: 이름=, 호불호: {}

[2] 사용자: 내 이름은 감귤이야
✅ 응답확인: {
  "response": "안녕하세요, 감귤님! 만나서 반갑습니다.",
  "new_name": "감귤",
  "new_likes": [],
  "new_dislikes": []
}
[2] 챗봇: 안녕하세요, 감귤님! 만나서 반갑습니다.
메모리: 이름=감귤, 호불호: {}

[3] 사용자: 김치찜을 좋아해
✅ 응답확인: {
  "response": "좋아하는 음식으로 김치찜을 추가했어요!",
  "new_name": null,
  "new_likes": ["김치찜"],
  "new_dislikes": []
}
[3] 챗봇: 좋아하는 음식으로 김치찜을 추가했어요!
메모리: 이름=감귤, 호불호: {'likes': ['김치찜']}

[4] 사용자: 해삼은 싫어해
✅ 응답확인: {
  "response": "해삼을 싫어하는군요. 알겠어요!",
  "new_name": null,
  "new_likes": [],
  "new_dislikes": ["해삼"]
}
[4] 챗봇: 해삼을 싫어하는군요. 알겠어요!
메모리: 이름=감귤, 호불호: {'likes': ['김치찜'], 'dislikes': ['해삼']}

[5] 사용자: 내 이름이 뭐였지?
✅ 응답확인: {
  "response": "당신의 이름은 감귤입니다.",
  "new_name": null,
  "new_likes": [],
  "new_dislikes": []
}
[5] 챗봇: 당신의 이름은 감귤입니다.


# 🟦 메모리 사용한 챗봇

# 상태 정의
TypedDict 사용

In [5]:
# MessagesState 와 같은 상태 만들어 보기
class State(TypedDict):
    messages: Annotated[list[str],  add_messages]

# 그래프 정의

In [6]:
def create_graph():
    graph_builder = StateGraph(State)

    memory = InMemorySaver()
    model = ChatOpenAI(model="gpt-4o")

    # Node 정의
    def generate(state: State):
        return {"messages": [model.invoke(state['messages'])]}

    # Node 추가
    graph_builder.add_node("generate", generate)

    # Edge 연결
    graph_builder.add_edge(START, "generate")
    graph_builder.add_edge("generate", END)  

    return graph_builder.compile(checkpointer=memory)


# 실행

In [11]:
config = {"configurable": {"thread_id": "abcd"}}

graph = create_graph()

while True:
    user_input = input("You\t:")

    if user_input.lower() in ["exit", "quit", "q"]:
        break

    for chunk, _ in graph.stream({"messages": [HumanMessage(user_input)]},
                 config,  # ✅ config (thres_id)
                 # stream_mode="values",):
                 stream_mode="messages",):  # 토큰 단위 스트리밍
        # event['messages'][-1].pretty_print()
        if chunk.content:
            print(chunk.content, end="", flush=True)
    
    # print(f"\n현재 메세지 개수: {len(event['messages'])}\n----------------------\n")
        

You	: Python 프로그래밍 언어에 대해 설명해봐


Python은 고수준의 프로그래밍 언어로, 읽기 쉽고 간결한 문법 덕분에 초보자와 전문가 모두에게 인기 있는 언어입니다. 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 발표되었습니다. Python은 다양한 용도로 사용되며 다음과 같은 특징과 장점을 가지고 있습니다.

1. **읽기 쉬운 문법**: Python은 코드의 가독성을 높이기 위해 설계되었습니다. 이는 개발자가 코드를 이해하고 유지보수하는 데 드는 시간을 줄여줍니다.

2. **광범위한 라이브러리 지원**: Python은 과학 계산, 데이터 분석, 웹 개발, 자동화 등 다양한 분야에서 사용되는 풍부한 표준 라이브러리와 서드파티 라이브러리를 제공합니다.

3. **다중 패러다임 지원**: Python은 객체 지향, 절차적, 함수형 프로그래밍을 포함한 여러 프로그래밍 패러다임을 지원합니다.

4. **인터프리터 언어**: Python은 인터프리터 언어로, 코드를 실행하기 전에 컴파일할 필요가 없습니다. 이는 개발 및 테스트 속도를 높이며, 플랫폼 간 이식성을 제공합니다.

5. **동적 타이핑**: Python은 동적 타이핑을 지원하므로, 변수의 타입을 명시적으로 선언할 필요가 없습니다. 덕분에 빠르고 유연한 코딩이 가능합니다.

6. **크로스 플랫폼**: Python은 Windows, macOS, 리눅스 등 다양한 운영체제에서 실행 가능합니다.

7. **강력한 커뮤니티**: Python은 커뮤니티가 매우 활발하며, 많은 문서와 온라인 지원이 제공됩니다. 새로운 도구와 라이브러리도 지속적으로 개발되고 있습니다.

이러한 이유들로 인해 Python은 웹 개발, 데이터 과학, 인공지능, 기계 학습, 자동화 및 기타 다양한 분야에서 널리 사용됩니다.

You	: q
